# Illustration Scrapping Studio — Jupyter
Runs the complete app on this server. Your browser only controls and reviews it. No training integration.

## Order of cells
1. **Configuration** — set `APP` to the project directory (the one containing `studio.py`).
2. **Setup/update** — stops the app, pulls source, installs pinned dependencies, and refreshes the UI. Rerun this after every update.
3. **Start once** — launches the app and prints the link to open.
4. **Stop** — graceful shutdown, only when you want it.
5. **Diagnose** — explains a start failure, such as an occupied port.

## The UI is not updated by `git pull`
`frontend/dist` is git-ignored, so pulling source never replaces the built UI. With Node installed, setup rebuilds it automatically. Without Node, build on your laptop with `npm ci` and `npm run build` in `frontend`, then upload `frontend-ui.zip` to the server and point `UI_ZIP` at it. The setup cell unpacks it into the exact folder the backend serves, so a fresh upload always takes effect.

Do not upload your laptop's `.venv`, credentials, or library as part of the source code.

In [ ]:
from pathlib import Path
import os, sys, subprocess, shutil, tempfile
APP = Path('/home/jovyan/Illustration-Scrapping-Studio').resolve()  # EDIT to the actual project directory
LIBRARY = Path('/home/jovyan/illustration-library').resolve()  # keep on persistent storage
UI_ZIP = Path('/home/jovyan/frontend-ui.zip').resolve()  # optional; only used when Node is absent
DIST = APP / 'frontend' / 'dist'  # the exact folder the backend serves
PORT = 8000
assert (APP / 'studio.py').is_file(), 'Set APP to the directory containing studio.py'
assert sys.version_info >= (3, 12), 'Python 3.12+ required'
assert os.environ.get('JUPYTERHUB_SERVICE_PREFIX'), 'Run inside your authenticated JupyterHub session'
BASE_PATH = os.environ['JUPYTERHUB_SERVICE_PREFIX'].rstrip('/') + f'/proxy/{PORT}'
print('Persistent library:', LIBRARY)
print('Served UI folder:  ', DIST)
print('Browser path:', BASE_PATH + '/')

In [ ]:
# Setup/update: stop the app, pull source, install deps, and refresh the built UI.
# Safe to rerun. Without Node, the uploaded UI_ZIP is unpacked into the served folder.
has_npm = bool(shutil.which('npm'))


def stop_app(grace=60):
    """Stop the app this notebook started, escalating if it refuses to exit."""
    process = globals().get('studio_process')
    if process is None or process.poll() is not None:
        print('No app started by this session is running.')
        return
    import signal
    process.send_signal(signal.SIGTERM)
    try:
        process.wait(timeout=grace)
        print('App stopped gracefully.')
    except subprocess.TimeoutExpired:
        process.kill()
        process.wait()
        print('Warning: app ignored the graceful stop and was killed. Check launcher.log.')


def refresh_ui(zip_path=UI_ZIP):
    """Replace the served frontend/dist with the uploaded build.

    Extracts through a staging folder so a bad zip cannot clear the working UI,
    and removes the old hashed assets so stale files cannot be served.
    """
    assert zip_path.is_file(), f'Upload the built frontend-ui.zip first: {zip_path}'
    with tempfile.TemporaryDirectory() as staging:
        shutil.unpack_archive(str(zip_path), staging)
        built = Path(staging) / 'dist'
        assert (built / 'index.html').is_file(), 'Zip must contain dist/index.html'
        if DIST.exists():
            shutil.rmtree(DIST)
        DIST.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(built), str(DIST))
    print('UI replaced. Now serving:', sorted(p.name for p in DIST.iterdir()))


stop_app()
subprocess.run(['git', '-C', str(APP), 'pull', '--ff-only', 'origin', 'main'], check=True)
command = [sys.executable, str(APP / 'studio.py'), 'setup']
if not has_npm:
    command.append('--skip-ui-build')
subprocess.run(command, cwd=APP, check=True)
if not has_npm:
    refresh_ui()
print('Update complete. Run the "Start once" cell next.')

In [ ]:
# Start once. A separate process continues when this browser disconnects.
# Server/container shutdown still stops it; restart explicitly after that.
import time, urllib.request, json
from IPython.display import display, HTML
from html import escape
if 'studio_process' in globals() and studio_process.poll() is None:
    raise RuntimeError('This notebook already started a running app')
LIBRARY.mkdir(parents=True, exist_ok=True)
launch_log = LIBRARY / 'launcher.log'
environment = dict(os.environ)
# Prevent inherited paths from silently targeting another library/database.
for key in ('ARTIST_DB_PATH', 'ARTIST_LIBRARY_PATH', 'ARTIST_LOG_PATH', 'ARTIST_PROVIDER_SETTINGS_PATH'):
    environment.pop(key, None)
with launch_log.open('ab') as output:
    studio_process = subprocess.Popen([str(APP / '.venv/bin/python'), str(APP / 'studio.py'), 'run', '--library', str(LIBRARY), '--database', str(LIBRARY / 'index.db'), '--port', str(PORT), '--base-path', BASE_PATH, '--no-browser'], cwd=APP, env=environment, stdout=output, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL, start_new_session=True)
for attempt in range(120):
    if studio_process.poll() is not None:
        tail = launch_log.read_text(errors='replace')[-2000:] if launch_log.is_file() else '(no launcher.log)'
        raise RuntimeError(f'App exited. The reason is the last line of launcher.log:\n\n{tail}')
    try:
        with urllib.request.urlopen(f'http://127.0.0.1:{PORT}/api/health', timeout=1) as response:
            health = json.load(response)
        if health['library_path'] != str(LIBRARY):
            raise RuntimeError('A different app already owns this port. Choose another PORT and rerun configuration.')
        time.sleep(1)
        if studio_process.poll() is not None:
            tail = launch_log.read_text(errors='replace')[-2000:] if launch_log.is_file() else '(no launcher.log)'
            raise RuntimeError(f'App exited. The reason is the last line of launcher.log:\n\n{tail}')
        display(HTML(f'<a href="{escape(BASE_PATH + "/", quote=True)}" target="_blank" rel="noopener">Open Illustration Scrapping Studio</a>'))
        break
    except OSError:
        time.sleep(0.5)
else:
    raise RuntimeError('Startup not ready. Read launcher.log; the process may still be starting.')

In [ ]:
# Stop only the process started by THIS notebook session. Run when needed.
# Existing job recovery applies when the backend starts again.
STOP_APP = False  # Set True only when you want to stop, then run this cell.
if not STOP_APP:
    print('App left running. Set STOP_APP=True to request shutdown.')
else:
    import signal
    if 'studio_process' in globals() and studio_process.poll() is None:
        stop_app()  # defined in the setup/update cell; escalates to kill if needed
        print('Shutdown requested. Active work may take time to finish; inspect launcher.log.')
        studio_process = None  # clear the handle so the start cell can run again
    else:
        print('No running process owned by this notebook session.')

## Reconnect and maintenance
Reopen the printed proxy link to reconnect. If the kernel restarts, do not blindly launch a second copy: the backend may still be running. Library locks and port checks reject duplicate starts. Use your terminal/process manager to stop the known app if the notebook has lost its process handle.

Provider credentials entered/uploaded through this UI are stored on the server. Use the authenticated Jupyter link; do not put this port behind a public anonymous tunnel. Back up privately with `studio.py backup` after stopping the backend; backups include credentials. See `docs/JUPYTER.md` for setup, updates and troubleshooting.


In [ ]:
# Diagnose a start failure: who holds the port, and what did the log say?
import socket, subprocess as sp
probe = socket.socket()
try:
    probe.bind(('127.0.0.1', PORT))
    print(f'Port {PORT} is free. Any earlier app has stopped.')
except OSError as exc:
    print(f'Port {PORT} is IN USE: {exc}\n'
          'An earlier app is still running and the new one exited.\n'
          'Run the stop cell with STOP_APP=True. If the notebook lost its process\n'
          'handle (for example after a kernel restart), find it in a terminal:\n'
          f'    pkill -f "studio.py run --port {PORT}"\n'
          'then rerun the start cell. To run alongside it, raise PORT in the configuration cell.')
finally:
    probe.close()
log = LIBRARY / 'launcher.log'
if log.is_file():
    print('\n--- last launcher.log lines ---')
    print(log.read_text(errors='replace')[-1500:])